# Documentação — Notebook 03_analise_clientes

Este notebook faz a **análise exploratória (EDA) da tabela `ecommerce_clientes`** lida diretamente do SQL Server via JDBC. Valida a estrutura, o volume e a integridade dos dados.

**Objetivo:** Entender o schema e a qualidade dos dados antes de processá-los no pipeline Bronze/Silver.

## Célula 1 — Importação

```python
from dotenv import load_dotenv
from pathlib import Path
import os
from pyspark.sql import functions as F
```

**O que faz:** Importa as bibliotecas necessárias:
- `load_dotenv` + `Path` + `os` — para carregar credenciais do `.env`
- `F` (pyspark.sql.functions) — funções Spark para filtragem, agregação, etc.

## Célula 2 — Carregamento do .env e definição da tabela

```python
load_dotenv(env_path)
sql_host = os.getenv("SQL_HOST")
...
tabela_destino = "squad2.ecommerce_clientes"
```

**O que faz:** Carrega o `.env` com as credenciais do SQL Server e define qual tabela será analisada (`squad2.ecommerce_clientes`).

## Célula 3 — Leitura do SQL Server via JDBC

```python
df_clientes_analise = spark.read \
    .format("sqlserver") \
    .option("host", sql_host) \
    .option("port", 1433) \
    .option("database", sql_database) \
    .option("dbtable", tabela_destino) \
    .option("user", sql_username) \
    .option("password", sql_password) \
    .load()
```

**O que faz:** Lê a tabela `squad2.ecommerce_clientes` do SQL Server usando o conector JDBC do Spark. Cria o DataFrame `df_clientes_analise` para análise.

**No serverless:** Esta operação funciona perfeitamente porque o JDBC do Spark usa o conector nativo do Databricks, sem necessidade de acesso direto à rede externa.

## Célula 4 — Estrutura e amostragem

```python
df_clientes_analise.printSchema()
display(df_clientes_analise.limit(10))
```

**O que faz:** Mostra o schema (tipos de cada coluna) e as 10 primeiras linhas dos dados. Permite uma primeira inspeção visual.

**Schema esperado:**
```
root
 |-- id_cliente: long
 |-- uuid_cliente: string
 |-- nome: string
 |-- sobrenome: string
 |-- email: string
 |-- senha_hash: string
 |-- dt_cadastro: timestamp
 |-- dt_ultima_atualizacao: timestamp
```

## Célula 5 — Volume de dados

```python
quantidade_linhas = df_clientes_analise.count()
quantidade_colunas = len(df_clientes_analise.columns)
```

**O que faz:** Conta o total de linhas e colunas do DataFrame. Serve para validar se o volume de dados está correto após a carga.

**Resultado esperado:** ~384 linhas, 8 colunas (pode variar conforme atualizações em tempo real).

## Célula 6 — Integridade da chave primária

```python
nulos_id_cliente = df_clientes_analise.filter(
    F.col("id_cliente").isNull()
).count()
```

**O que faz:** Verifica se existe algum registro com `id_cliente` nulo. A chave primária não pode ter nulos — se houver, indica problema na origem.

**Resultado esperado:** `0` (saudável) — nenhum registro com id_cliente nulo.

## Resumo do fluxo

| Célula | Função |
| --- | --- |
| 1 | Importa bibliotecas (dotenv, F) |
| 2 | Carrega .env e define tabela |
| 3 | Lê tabela do SQL Server via JDBC |
| 4 | Mostra schema e amostra dos dados |
| 5 | Conta linhas e colunas |
| 6 | Valida chave primária (sem nulos) |

**Fluxo:** Importa → carrega credenciais → lê → inspeciona → valida

**Este notebook responde:** Como são os dados? Quantos registros? A chave primária está íntegra?